## Financial Engineering Project 
### Project Description 
- Collect Weekly Data for the top $p=400$ stocks of the S&P500 over $n=26$ weeks
- Compute the Weekly Excess Returns Matrix $M$ (p x n)
- Subtract the mean from each row for de-meaned excess returns matrix $Y$ (p x n)
- Create a single factor market model for the covariance matrix $\Sigma$ (p x p)
- Compute the Global minimum variance portfolio holding vector
- Compute the portfolio variance, std deviation, beta
- Compute individual stock variances, std deviation, and beta

In [1]:
import numpy as np
import requests, pandas as pd
import yfinance as yf
from io import StringIO

### 1) Web Scrape for Stock TickersUsing Pandas

In [2]:
API = 'https://en.wikipedia.org/w/api.php'

params = {
    'action': 'parse',
    'page': 'List_of_S&P_500_companies', 
    'prop': 'text', 
    'format': 'json'
}

headers = {'User-Agent': 'sp500-ticker-tutorial/1.0'}
r = requests.get(API, params = params, headers = headers)
html = r.json()['parse']['text']['*']

df = pd.read_html(StringIO(html))[0]
tickers = df['Symbol'].astype(str).tolist()
print("Number of stocks:", len(tickers))

Number of stocks: 503


We Now have a list of the S&P 500 stock tickers. The next thing to do is sort this list by market cap and return the top 400 members of the sorted lsit 

In [3]:
for bad in ["BRK.B", "BF.B"]:
    if bad in tickers:
        tickers.remove(bad)

tickers = tickers + ["BRK-B","BF-B"]
start_date = "2025-05-26"
end_date = "2025-11-28"
results = []

data = yf.download(tickers, start=start_date, end=end_date, interval="1wk")
closing_prices = data['Close']

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  503 of 503 completed


Now that we have data from yfinance download we need to organize it by mkt cap for the top 400 stocks 

In [4]:
for ticker in tickers:
    try:
        last_price = closing_prices[ticker].dropna().iloc[-1] #use the most recent close price
        stock = yf.Ticker(ticker)
        shares = stock.info.get('sharesOutstanding')

        if shares != None:
            m_cap = last_price * shares
            results.append({'Ticker': ticker, 'MarketCap': m_cap})
            
    except Exception: 
        continue 

dataframe = pd.DataFrame(results)
top_400 = dataframe.sort_values(by='MarketCap', ascending=False).head(400)

In [18]:
top_400_tickers = top_400["Ticker"].tolist() #Now we have our top 400 stocks list 
top_400_data = yf.download(top_400_tickers, start= start_date, end = end_date, interval = "1wk")
top_400_close = top_400_data["Close"]

[*********************100%***********************]  400 of 400 completed


In [20]:
missing_counts = top_400_close.isna().sum()
problems = missing_counts[missing_counts > 0]
print(problems)

Series([], dtype: int64)


This is nice that we have no missing values from our data, we can now proceed with the computations

In [24]:
returns = top_400_close.pct_change().dropna() # the second dropna is to ensure proper shape
returns_matrix = returns.T.to_numpy()
print("Matrix Dimensions:", returns_matrix.shape)

Matrix Dimensions: (400, 26)


Compute weekly excess returns matrix: $ExcessReturns = TotalReturns - RiskFreeRate$

In [29]:
risk_free_rate = 0.0409 # 10-yr treasury bonds 
weekly_RFR = risk_free_rate/52 
print("Risk Free Rate", risk_free_rate)
print("Weekly Risk Free Rate", weekly_RFR)

# Excess Returns Formula
excess_returns_matrix = returns_matrix - weekly_RFR 

Risk Free Rate 0.0409
Weekly Risk Free Rate 0.0007865384615384616


Compute mean vector $\mu$ and de-meaned weekly excess returns matrix: $Y = ExcessReturnsMatrix - \mu$

In [35]:
# axis=1 takes mean of each row and creates a column vector of each stocks mean 
# keepdims=True enable matrix subtraction by copying each collumn to match the matrix being worked on  
excess_returns_mean_vector = excess_returns_matrix.mean(axis=1, keepdims= True)

# Creating the p x n de-meaned excess returns matrix Y
Y = excess_returns_matrix - excess_returns_mean_vector

Compute weekly sample Covariance Matrix: $ S = YY^T/n$ 

In [65]:
S = (Y @ Y.T) / 26 
print("Matrix Dimensions:", S.shape)

eigenvalues, eigenvectors = np.linalg.eig(S)
index = np.argmax(eigenvalues)

lambda_sq = eigenvalues[index]
h = eigenvectors[:,index].reshape(-1,1)

print("leading/largest eigenvalue :", lambda_sq)
print("Correspoding Eigenvector Dimensions: ", h.shape)

Matrix Dimensions: (400, 400)
leading/largest eigenvalue : (0.13084459299471166+0j)
Correspoding Eigenvector Dimensions:  (400, 1)


Now that we have the necessatu components, we can create the single factor market model for the covariance matrix: $\Sigma = (\lambda^2 - \ell^2)h h^T + (n/p)\ell^2 I$ where $\ell^2 = \frac{tr(S) - \lambda^2}{n -1} $

In [74]:
trace_S = np.trace(S)
ell_sq = (trace_S - lambda_sq) / (25)

term1 = (lambda_sq - ell_sq) * (h @ h.T)
term2 = (26/400) * ell_sq * np.eye(400)
Sigma = term1 + term2

print("Matrix Dimension: ", Sigma.shape)

Matrix Dimension:  (400, 400)


Compute global-minimum-variance portfolio holding vector: $h_C = S^{-1}e / e^TSe$ where $e = (1,1,...,1)$

In [77]:
# Compute Global-minimum-Variance Pfolio C 
e = np.ones(400, dtype=float) # characteristic of C 
Sigma_inverse = np.linalg.inv(Sigma) # Inverse Covariance Matrix

# Holding pfolio for C 
holdings_C = (Sigma_inverse @ e) / (e.T @ Sigma_inverse @ e)

print("Total Holdings Vector Sum (fully invested)", sum(holdings_C))

Total Holdings Vector Sum (fully invested) (0.9999999999999991+0j)


Compute GMV portfolio variance $\sigma^2_C = h^T_C S h_C$

In [78]:
# Calculate the variance 
Variance_C = (holdings_C.T @ Sigma @ holdings_C)
annual_Variance_C = Variance_C * 52 
print("Pfolio C Variance (annual): ", annual_Variance_C)

Pfolio C Variance (annual):  (0.00047786300417193686+0j)


Compute GMV portfolio standard deviation: $\sigma = \sqrt(h^T_C S h_C)$

In [79]:
# Compute Standard Deviation 
std_pfolo_C = np.sqrt(Variance_C)
annualized_std_pfolio_C = std_pfolo_C * np.sqrt(52)

print("Portfolio C Standard Deviation (annual): ", annualized_std_pfolio_C)

Portfolio C Standard Deviation (annual):  (0.021860077862897394+0j)


Compute expected excess returns: $f_C = E[r_C] = h^T_C  \mu$